In [1]:
# 1. 사전 환경 준비
# !pip install shap wandb optuna-integration python-dotenv

In [2]:
import os
import wandb
from dotenv import load_dotenv

# 2. W&B 로그인 (env 파일에 저장된 API 키 로드)
load_dotenv()
wandb_key = os.getenv("WANDB_API_KEY")
if wandb_key and wandb_key != "your_wandb_api_key_here":
    wandb.login(key=wandb_key)
    print("W&B Logged in successfully!")
else:
    print("W&B API key not found in .env. Please set it!")
    wandb.login()  # Interactive prompt fallback

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/admin/.netrc
wandb: Currently logged in as: genie0320 (genie0320-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B Logged in successfully!


In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from scipy.special import logit, expit

data_dir = "." if os.path.exists("train.csv") else "../.."
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))
print("Train shape:", train.shape, "Test shape:", test.shape)

Train shape: (3000, 18) Test shape: (3000, 17)


In [4]:
# ==========================================
# 3. 결측치 처리 및 Data-Centric 보강
# ==========================================
for df in [train, test]:
    df["medical_history"] = df["medical_history"].fillna("none")
    df["family_medical_history"] = df["family_medical_history"].fillna("none")
    df["edu_level"] = df["edu_level"].fillna("Unknown")

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# IterativeImputer를 사용하여 수치형 변수 대치 (mean_working 제외)
numeric_cols_for_imputation = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

imputer = IterativeImputer(random_state=42, max_iter=20)

# Data Leakage 방지를 위해 Train 데이터로만 Imputer를 학습시킴
train[numeric_cols_for_imputation] = imputer.fit_transform(
    train[numeric_cols_for_imputation]
)
test[numeric_cols_for_imputation] = imputer.transform(test[numeric_cols_for_imputation])

# mean_working 결측치는 -1로 대치하여 트리가 결측 자체를 인식하게 유도
train["mean_working"] = train["mean_working"].fillna(-1)
test["mean_working"] = test["mean_working"].fillna(-1)

print("Imputation complete.")

Imputation complete.


In [5]:
# ==========================================
# 4. 파생 변수 추가 (원본 변수는 절대 삭제하지 않음)
# ==========================================
for df in [train, test]:
    # ag가 누락했던 결측치 시그널 파생 변수 강제 추가
    df["is_working_missing"] = (df["mean_working"] == -1.0).astype(int)
    # disease_count 파생 변수 (희소성 보완)
    df["medical_disease_count"] = df["medical_history"].apply(
        lambda x: 0 if x == "none" else len(str(x).split(","))
    )
    df["family_disease_count"] = df["family_medical_history"].apply(
        lambda x: 0 if x == "none" else len(str(x).split(","))
    )

    # 비선형 생체 지표 도메인 파생 변수 추가
    df["bmi"] = df["weight"] / ((df["height"] / 100) ** 2)
    df["pulse_pressure"] = (
        df["systolic_blood_pressure"] - df["diastolic_blood_pressure"]
    )
    df["map"] = (df["systolic_blood_pressure"] + 2 * df["diastolic_blood_pressure"]) / 3
    df["glucose_cholesterol_ratio"] = df["glucose"] / (df["cholesterol"] + 1)
    df["overwork_and_poor_sleep"] = (
        (df["mean_working"] >= 12) & (df["sleep_pattern"] == "sleep difficulty")
    ).astype(int)
    df["vascular_bone_risk"] = (
        (df["bone_density"] <= -1.0) & (df["pulse_pressure"] > 80)
    ).astype(int)

# 복합 질환(medical_history) 동적 이진화 플래그 생성 (train 기준으로 추출)
diseases = set()
for col in ["medical_history", "family_medical_history"]:
    for val in train[col].dropna().unique():
        for d in val.split(","):
            diseases.add(d.strip())
diseases.discard("none")
diseases = sorted(list(diseases))

for col, prefix in [("medical_history", "med"), ("family_medical_history", "fam")]:
    for disease in diseases:
        feat_name = f'{prefix}_{disease.lower().replace(" ", "_")}'
        train[feat_name] = train[col].apply(
            lambda x: 1 if disease in [d.strip() for d in x.split(",")] else 0
        )
        test[feat_name] = test[col].apply(
            lambda x: 1 if disease in [d.strip() for d in x.split(",")] else 0
        )

# Ordinal Encoding 매핑
edu_map = {
    "Unknown": 0,
    "high school diploma": 1,
    "bachelors degree": 2,
    "graduate degree": 3,
}
activity_map = {"light": 1, "moderate": 2, "intense": 3}
for df in [train, test]:
    # ag가 누락했던 결측치 시그널 파생 변수 강제 추가
    df["is_working_missing"] = (df["mean_working"] == -1.0).astype(int)
    df["edu_level_encoded"] = df["edu_level"].map(edu_map)
    df["activity_encoded"] = df["activity"].map(activity_map)

# 문자열 데이터는 Label Encoding
categorical_cols = [
    "gender",
    "smoke_status",
    "sleep_pattern",
    "activity",
    "edu_level",
    "medical_history",
    "family_medical_history",
]
for col in categorical_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]]).astype(str))
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

print("Feature Engineering Complete. Train shape:", train.shape)

Feature Engineering Complete. Train shape: (3000, 35)


In [6]:
x_train = train.drop(["ID", "stress_score"], axis=1)
y_train = train["stress_score"]
x_test = test.drop("ID", axis=1)

In [ ]:
# ==========================================
# 5. SHAP 기반 노이즈 제어 (최하위 3개 제명)
# ==========================================
import shap
import lightgbm as lgb
import matplotlib.pyplot as plt

print("Calculating SHAP values for baseline model feature selection...")

# Baseline LightGBM 학습
baseline_model = lgb.LGBMRegressor(
    objective="regression_l1", random_state=42, verbose=-1, n_jobs=1
)
baseline_model.fit(x_train, y_train)

# SHAP TreeExplainer 기여도 계산
explainer = shap.TreeExplainer(baseline_model)
shap_values = explainer.shap_values(x_train)

if isinstance(shap_values, list):
    shap_avg = np.abs(shap_values[0]).mean(axis=0)
else:
    shap_avg = np.abs(shap_values).mean(axis=0)

shap_imp = pd.Series(shap_avg, index=x_train.columns).sort_values(ascending=False)
print("\n--- SHAP Feature Importances (Absolute Mean) ---")
for feat, val in shap_imp.items():
    print(f"{feat}: {val:.6f}")

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, x_train, plot_type="bar", show=False)
plt.tight_layout()
output_dir = "result/v10" if os.path.exists("result/v10") else "."
os.makedirs(output_dir, exist_ok=True)
plt.savefig(os.path.join(output_dir, "shap_summary_plot_v10.png"))
plt.close()

# 최하위 20% 피처 대규모 자동 제거 (소심한 N=3 방기)
drop_ratio = 0.2
N = int(len(shap_imp) * drop_ratio)
features_to_drop = list(shap_imp.tail(N).index)
print(f"\nRemoving bottom {N} features from dataset: {features_to_drop}")
x_train = x_train.drop(
    columns=[f for f in features_to_drop if f in x_train.columns], errors="ignore"
)
x_test = x_test.drop(
    columns=[f for f in features_to_drop if f in x_test.columns], errors="ignore"
)

Calculating SHAP values for baseline model feature selection...

--- SHAP Feature Importances (Absolute Mean) ---
mean_working: 0.021534
cholesterol: 0.021037
height: 0.020218
diastolic_blood_pressure: 0.016971
weight: 0.016383
glucose: 0.015571
map: 0.015561
bmi: 0.013650
glucose_cholesterol_ratio: 0.013382
bone_density: 0.013006
systolic_blood_pressure: 0.008798
pulse_pressure: 0.008215
smoke_status: 0.007798
age: 0.006986
edu_level: 0.006209
activity: 0.004682
sleep_pattern: 0.004525
family_disease_count: 0.004394
edu_level_encoded: 0.004289
medical_disease_count: 0.002637
family_medical_history: 0.002288
fam_diabetes: 0.002101
med_high_blood_pressure: 0.001804
gender: 0.001672
med_heart_disease: 0.001637
medical_history: 0.001472
med_diabetes: 0.001360
fam_heart_disease: 0.000607
fam_high_blood_pressure: 0.000208
is_working_missing: 0.000000
vascular_bone_risk: 0.000000
overwork_and_poor_sleep: 0.000000
activity_encoded: 0.000000

Removing bottom 6 features from dataset: ['fam_hear

In [8]:
# ==========================================
# 6. Optuna Tuning (Original Target Space with robust regularization)
# ==========================================
import optuna
from optuna_integration.wandb import WeightsAndBiasesCallback
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import warnings

warnings.filterwarnings("ignore")


def tune_xgboost(x_train, y_train):

    def objective(trial):
        params = {
            "objective": "reg:absoluteerror",
            "random_state": 42,
            "verbosity": 0,
            "n_jobs": 1,
            "enable_categorical": True,
            "tree_method": "hist",
            "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.005, 0.05, log=True
            ),
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "min_child_weight": trial.suggest_int("min_child_weight", 2, 20),
            "subsample": trial.suggest_float("subsample", 0.5, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
            "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        }

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mae_scores, mse_scores, r2_scores = [], [], []

        for train_idx, val_idx in kf.split(x_train, y_train):
            X_tr, y_tr = x_train.iloc[train_idx], y_train.iloc[train_idx]
            X_val, y_val = x_train.iloc[val_idx], y_train.iloc[val_idx]

            model = xgb.XGBRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

            preds = model.predict(X_val)
            # Clip predictions within [0, 1] range as per task bounds
            preds_clipped = np.clip(preds, 0.0, 1.0)

            mae_scores.append(mean_absolute_error(y_val, preds_clipped))
            mse_scores.append(mean_squared_error(y_val, preds_clipped))
            r2_scores.append(r2_score(y_val, preds_clipped))

        mean_mae = np.mean(mae_scores)
        mean_mse = np.mean(mse_scores)
        mean_rmse = np.sqrt(mean_mse)
        mean_r2 = np.mean(r2_scores)

        trial.set_user_attr("mse", mean_mse)
        trial.set_user_attr("rmse", mean_rmse)
        trial.set_user_attr("r2", mean_r2)

        return mean_mae

    wandb_kwargs = {
        "project": "stress_index_v10",
        "name": "xgboost_v10_tuning",
        "reinit": True,
    }
    wandbc = WeightsAndBiasesCallback(metric_name="mae", wandb_kwargs=wandb_kwargs)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=100, callbacks=[wandbc])
    wandb.finish()
    return study.best_params


print("Tuning XGBoost Hyperparameters...")
best_xgb = tune_xgboost(x_train, y_train)
print(f"Best XGBoost Params: {best_xgb}")

Tuning XGBoost Hyperparameters...


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[I 2026-06-17 12:05:52,262] A new study created in memory with name: no-name-e8e9db3f-b855-4370-9428-9b4c40256d67
[I 2026-06-17 12:05:53,413] Trial 0 finished with value: 0.23258296879529952 and parameters: {'n_estimators': 336, 'learning_rate': 0.021436081516806416, 'max_depth': 7, 'min_child_weight': 15, 'subsample': 0.6992300748375405, 'colsample_bytree': 0.5914492731646313, 'gamma': 4.662956147845546, 'alpha': 0.01570465556389551, 'lambda': 2.4607067977225996}. Best is trial 0 with value: 0.23258296879529952.
[I 2026-06-17 12:06:00,008] Trial 1 finished with value: 0.19514346943875155 and parameters: {'n_estimators': 486, 'learning_rate': 0.016465477477065716, 'max_depth': 11, 'min_child_weight': 2, 'subsample': 0.8933346632281864, 'colsample_bytree': 0.7440977150764413, 'gamma': 3.01177248925844, 'alpha': 0.0020541160906465813, 'lambda': 3.0736156657819733}. Best is trial 1 with value: 0.19514346943875155.
[I 2026-06-17 12:06:12,170] Trial 2 finished with value: 0.1826602510592589

alpha,▁▁▁▅▂▁▁▁▂█▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▂▁▁▁▁▂▄▂▃▂▁▁▁▁▁
colsample_bytree,▂▆▄▂█▆▆▇█▄▄▁▅▅▆▄▅▆▆▅▆▆▆█▆▇▆▇▅██▇███▇▇▆▆▇
gamma,▆▃▆▅█▃▄▃▃▅▂▂▃▂▁▂▃▃▃▃▄▄▃▂▃▂▂▂▃▂▂▂▂▁▂▁▁▁▂▁
lambda,▄▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇█▆▅▅▄▃
learning_rate,▇▃▅▄▁▅▁▇▃▅▄▄▄▅▅▄▂▅▆▆█▇▆▆▆▇▇▆▆▆▅▇█▆▅▄▄▄▄▃
mae,▄▆▄█▅▃▄▃▅▄▅▃▃█▄▃▄▄▃▃▄▃▂▂▂▂▃▂▄▂▄▂▁▁▁▁▁▁▁▂
max_depth,▅▄▅▁▇█▁█▆▅▇▆▆▅▇▇▆█▆▇▅▅▄▄▃▆▆▆▆▆▆▆▇███████
min_child_weight,▆▇▅▆▁█▁▃█▂▂▃▁▁▁▁▂▁▆▂▂▁▁▂▂▁▁▁▁▇▁▂▁▁▁▁▁▁▁▂
n_estimators,▁▆▆▄▅▇▃▆▆▅▄▃▂▆▇▇█▇█▇▆█▇▆▆▇▇▇▆▆▇█▇▇█▇▇▇▇█
subsample,█▄▂█▃▆▁▇▃▅█▄▅▄▆▅▅▆▆▄▄▄▃▃▃▅▆▅▆▄▇▅▅▆▆▆▆▆▆▆
alpha,0.04428


Best XGBoost Params: {'n_estimators': 928, 'learning_rate': 0.02545527924057087, 'max_depth': 12, 'min_child_weight': 2, 'subsample': 0.8121229984836017, 'colsample_bytree': 0.800500825455789, 'gamma': 0.27255111804653137, 'alpha': 0.05781588842701614, 'lambda': 5.460278840204996}


In [9]:
# ==========================================
# 7. Seed Averaging on Target Space
# ==========================================
def train_with_seeds(x_train, y_train, x_test, best_params, seeds=[42, 2026, 777]):
    oof_preds = np.zeros(len(x_train))
    test_preds = np.zeros(len(x_test))
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(kf.split(x_train, y_train)):
        X_tr, y_tr = x_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
        X_val, y_val = x_train.iloc[val_idx].copy(), y_train.iloc[val_idx]

        fold_val_preds = np.zeros(len(X_val))
        fold_test_preds = np.zeros(len(x_test))

        for seed in seeds:
            params = best_params.copy()
            params.update(
                {
                    "objective": "reg:absoluteerror",
                    "random_state": seed,
                    "verbosity": 0,
                    "enable_categorical": True,
                    "tree_method": "hist",
                    "n_jobs": 1,
                }
            )

            model = xgb.XGBRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

            fold_val_preds += np.clip(model.predict(X_val), 0.0, 1.0) / len(seeds)
            fold_test_preds += np.clip(model.predict(x_test), 0.0, 1.0) / len(seeds)

        oof_preds[val_idx] = fold_val_preds
        test_preds += fold_test_preds / 5

    mae = mean_absolute_error(y_train, oof_preds)
    print(f"[XGBoost Single] Seed Averaged OOF MAE: {mae:.6f}")
    return oof_preds, test_preds


oof_xgb, test_xgb = train_with_seeds(x_train, y_train, x_test, best_xgb)

final_oof_clipped = np.clip(oof_xgb, 0.0, 1.0)
print(f"Final Clipped OOF MAE: {mean_absolute_error(y_train, final_oof_clipped):.6f}")

test_preds = np.clip(test_xgb, 0.0, 1.0)

[XGBoost Single] Seed Averaged OOF MAE: 0.169968
Final Clipped OOF MAE: 0.169968


In [10]:
submission = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))
submission["stress_score"] = test_preds
output_dir = "result/v10" if os.path.exists("result/v10") else "."
os.makedirs(output_dir, exist_ok=True)
submission.to_csv(os.path.join(output_dir, "submit_10.csv"), index=False)
print("Submission saved to submit_10.csv")

Submission saved to submit_10.csv
